In [ ]:
%sql
DROP TABLE IF EXISTS workspace.gold_weather.kpi_energy_daily;

CREATE TABLE IF NOT EXISTS workspace.gold_weather.kpi_energy_daily (
  location_id BIGINT,
  date_key BIGINT,
  data_type STRING,
  solar_potential_mj_m2 DOUBLE,
  sunshine_hours DOUBLE,
  wind_power_class STRING
)

In [ ]:
# Bandas de wind_power_class inspiradas en la escala de Beaufort, no en la
# curva de potencia de un aerogenerador real, ver Proyecto/DECISIONS.md #6
kpi = spark.sql("""
    SELECT
        location_id,
        date_key,
        data_type,
        shortwave_radiation_sum AS solar_potential_mj_m2,
        sunshine_duration / 3600.0 AS sunshine_hours,
        CASE
            WHEN wind_speed_max < 12 THEN 'calmo'
            WHEN wind_speed_max < 28 THEN 'moderado'
            WHEN wind_speed_max < 50 THEN 'fuerte'
            ELSE 'muy_fuerte'
        END AS wind_power_class
    FROM workspace.gold_weather.fact_weather_daily
""")

kpi.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold_weather.kpi_energy_daily")

In [ ]:
display(kpi.limit(5))